# Logistic Regression - From Scratch và `scikit-learn`


## Lý thuyết Logistic Regression

**Logistic Regression** là một thuật toán phân loại tuyến tính cổ điển nhưng vô cùng mạnh mẽ, thường dùng cho các bài toán phân loại nhị phân (binary classification). Cụ thể:

### 1. Ý tưởng cốt lõi
* Ánh xạ tổ hợp tuyến tính của các feature $z = w^T x + b$ qua hàm **Sigmoid**: $g(z) = \frac{1}{1 + e^{-z}}$ để đưa đầu ra về khoảng $(0, 1)$.
* Đầu ra $a = g(z)$ biểu thị xác suất thuộc về lớp positive (lớp 1): $P(y=1|x)$.
* Phân lớp bằng cách so sánh xác suất $a$ với một ngưỡng quyết định (threshold), mặc định là 0.5.

### 2. Các bước thực hiện

* **Bước 1 (Tính giá trị dự đoán):** Tính giá trị tuyến tính $z = Xw + b$ và xác suất dự đoán tương ứng $a = \sigma(z)$.
* **Bước 2 (Tính Loss):** Sử dụng hàm mất mát **Binary Cross-Entropy Loss** để đo lường sai số:
  $$J(w, b) = -\frac{1}{m} \sum_{i=1}^m \left[ y^{(i)} \log(a^{(i)}) + (1 - y^{(i)}) \log(1 - a^{(i)}) \right]$$
* **Bước 3 (Gradient Descent):** Tính toán đạo hàm của hàm Loss theo trọng số và bias để cập nhật tham số:
  $$dw = \frac{1}{m} X^T (a - y)$$
  $$db = \frac{1}{m} \sum_{i=1}^m (a^{(i)} - y^{(i)})$$
  $$w \leftarrow w - \alpha \cdot dw$$
  $$b \leftarrow b - \alpha \cdot db$$

### 3. Ưu điểm của Logistic Regression
* Đơn giản, trực quan, dễ lập trình và huấn luyện rất nhanh.
* Trả về xác suất nên rất hữu ích trong các bài toán cần đánh giá mức độ tin cậy.
* Hoạt động rất tốt trên các tập dữ liệu phân tách tuyến tính và dễ dàng regularize (L1/L2) để tránh overfitting.


## Triển khai `LogisticRegression` không dùng thư viện

### Import thư viện và load dữ liệu

In [ ]:
from __future__ import annotations

import pandas as pd
import numpy as np
import joblib
import sklearn.linear_model
import os, sys, time

# Thêm đường dẫn thư mục gốc để import custom modules
sys.path.append(os.path.abspath(os.path.join('..')))
from practice_1.utils.custom_hyperparameter_tuning import CustomGridSearchCV
from practice_1.utils.custom_cv import CustomStratifiedKFold

# Load dữ liệu đã được xử lý ở bước Feature Engineering
X_train = joblib.load('./data/ready_for_train/X_train_final.pkl')
X_test  = joblib.load('./data/ready_for_train/X_test_final.pkl')
y_train = joblib.load('./data/ready_for_train/y_train.pkl')
y_test  = joblib.load('./data/ready_for_train/y_test.pkl')

print('Training dataset shape:')
print(f'X: {X_train.shape}')
print(f'y: {y_train.shape}')

print('Testing dataset shape:')
print(f'X: {X_test.shape}')
print(f'y: {y_test.shape}')


### Triển khai Logistic Regression không dùng `sklearn`

In [ ]:
class LogisticRegressionScratch:
    """Triển khai Logistic Regression không dùng thư viện sklearn."""

    def __init__(self, learning_rate: float = 0.1, epochs: int = 500):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def get_params(self, deep=True):
        return {
            "learning_rate": self.learning_rate,
            "epochs": self.epochs
        }
        
    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

    def sigmoid(self, z):
        # Áp dụng hàm kích hoạt Sigmoid đưa đầu ra tuyến tính về khoảng (0, 1)
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))

    def compute_loss(self, y_true, y_pred):
        # Tính toán Binary Cross-Entropy Loss (Log Loss)
        eps = 1e-15
        # Giới hạn giá trị của y_pred để tránh lỗi tràn số log(0)
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    def fit(self, X, y):
        y = np.array(y).flatten()
        n_samples, n_features = X.shape

        # Khởi tạo tham số trọng số (weights) và bias bằng 0
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.loss_history = []

        # Vòng lặp tối ưu hóa Gradient Descent
        for epoch in range(self.epochs):
            # Tính giá trị kết hợp tuyến tính z và xác suất dự đoán a
            linear = X.dot(self.weights) + self.bias
            pred = self.sigmoid(linear)

            # Tính toán đạo hàm riêng của Loss theo weights và bias
            dw = (1 / n_samples) * X.T.dot(pred - y)
            db = (1 / n_samples) * np.sum(pred - y)

            # Cập nhật tham số theo learning rate
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            # Lưu lại loss lịch sử để theo dõi độ hội tụ
            loss = self.compute_loss(y, pred)
            self.loss_history.append(loss)
            
        return self

    def predict_proba(self, X):
        # Trả về xác suất thuộc về lớp dương (lớp 1)
        linear = X.dot(self.weights) + self.bias
        return self.sigmoid(linear)

    def predict(self, X, threshold=0.5):
        # Dự đoán nhãn lớp (0 hoặc 1) dựa vào ngưỡng threshold chỉ định
        return (self.predict_proba(X) >= threshold).astype(int)


### Kiểm tra nhanh mô hình Logistic Regression From Scratch

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Kiểm tra nhanh mô hình tự xây dựng với các siêu tham số mặc định
quick_lr = LogisticRegressionScratch(learning_rate=0.1, epochs=500)

start = time.time()
quick_lr.fit(X_train, y_train)
train_time_scratch = time.time() - start

quick_pred = quick_lr.predict(X_test)
acc = accuracy_score(y_test, quick_pred)
prec = precision_score(y_test, quick_pred)
rec = recall_score(y_test, quick_pred)
f1 = f1_score(y_test, quick_pred)

print(f'Quick test (learning_rate=0.1, epochs=500) - Train time: {train_time_scratch:.1f}s')
print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1 Score : {f1:.4f}')


## Dùng thư viện `sklearn` để triển khai `LogisticRegression`

In [ ]:
from sklearn.linear_model import LogisticRegression as SklearnLR

# Huấn luyện mô hình từ thư viện sklearn với cấu hình tiêu chuẩn
sklearn_lr = SklearnLR(max_iter=1000, random_state=42)

start = time.time()
sklearn_lr.fit(X_train, y_train)
train_time_sklearn = time.time() - start

sklearn_pred = sklearn_lr.predict(X_test)

acc_sk = accuracy_score(y_test, sklearn_pred)
prec_sk = precision_score(y_test, sklearn_pred)
rec_sk = recall_score(y_test, sklearn_pred)
f1_sk = f1_score(y_test, sklearn_pred)

print(f'sklearn LR - Train time: {train_time_sklearn:.1f}s')
print(f'Accuracy : {acc_sk:.4f}')
print(f'Precision: {prec_sk:.4f}')
print(f'Recall   : {rec_sk:.4f}')
print(f'F1 Score : {f1_sk:.4f}')


## Hyperparameter Tuning

Để tối ưu hóa Logistic Regression, Grid Search kết hợp với **3-fold Cross Validation** được sử dụng nhằm tìm ra bộ hyperparameters phù hợp nhất.

Parameter grid được sử dụng:

```python
lr_grid = {
    'learning_rate': [0.1, 0.01],
    'epochs': [500, 1000],
}
```

Mô hình được đánh giá dựa trên hai độ đo chính là **F1-score** và **Accuracy** trên tập tập huấn luyện thông qua cross-validation.


In [ ]:
lr_grid = {
    'learning_rate': [0.1, 0.01],
    'epochs': [500, 1000],
}


### Grid Search dùng F1-score

In [ ]:
cv = CustomStratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scratch_lr = LogisticRegressionScratch()

# Thiết lập Grid Search tối ưu hóa F1-score
lr_grid_search_f1 = CustomGridSearchCV(
    estimator=scratch_lr,
    param_grid=lr_grid,
    cv=cv,
    scoring='f1'
)

lr_grid_search_f1.fit(X_train, y_train)


### Grid Search dùng Accuracy

In [ ]:
cv = CustomStratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scratch_lr = LogisticRegressionScratch()

# Thiết lập Grid Search tối ưu hóa Accuracy
lr_grid_search_acc = CustomGridSearchCV(
    estimator=scratch_lr,
    param_grid=lr_grid,
    cv=cv,
    scoring='accuracy'
)

lr_grid_search_acc.fit(X_train, y_train)


## Threshold Tuning để tối ưu False Positive Rate (FPR)

Trong các bài toán phân loại như lọc thư rác (Spam Detection), việc phân loại nhầm một email thường (Ham) thành thư rác (Spam) là một sai lầm rất nghiêm trọng (False Positive). Để giảm thiểu tỷ lệ này dưới mức mong muốn (ví dụ <= 1% FPR), ta có thể dịch chuyển ngưỡng quyết định (threshold) phân loại thay vì dùng ngưỡng mặc định 0.5.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Lấy mô hình scratch có tham số tốt nhất
best_scratch_model = lr_grid_search_f1.best_estimator_
probs = best_scratch_model.predict_proba(X_test)

# Tạo dải ngưỡng từ 0.5 đến 0.999 để đánh giá
candidate_thresholds = np.linspace(0.5, 0.999, 100)
target_threshold = 0.5
closest_fpr = 1.0

print(f"{'Threshold':<12}{'FP':<8}{'TN':<8}{'FPR (%)':<12}{'TPR (Recall) (%)':<15}")
print("-" * 55)

for th in candidate_thresholds:
    preds = (probs >= th).astype(int)
    
    tp_c = np.sum((y_test == 1) & (preds == 1))
    fp_c = np.sum((y_test == 0) & (preds == 1))
    tn_c = np.sum((y_test == 0) & (preds == 0))
    fn_c = np.sum((y_test == 1) & (preds == 0))
    
    fpr_c = fp_c / (fp_c + tn_c) if (fp_c + tn_c) > 0 else 0.0
    tpr_c = tp_c / (tp_c + fn_c) if (tp_c + fn_c) > 0 else 0.0
    
    # Chỉ hiển thị các ngưỡng đưa tỷ lệ báo sai FPR về khoảng chấp nhận được (0.5% - 4%)
    if 0.005 <= fpr_c <= 0.04:
        print(f"{th:<12.4f}{fp_c:<8}{tn_c:<8}{fpr_c*100:<12.2f}{tpr_c*100:<15.2f}")
        
    # Tìm ngưỡng tốt nhất tiệm cận mục tiêu FPR = 1%
    if abs(fpr_c - 0.01) < abs(closest_fpr - 0.01):
        closest_fpr = fpr_c
        target_threshold = th

print(f"Threshold tốt nhất: {target_threshold:.4f}")
print(f"FPR đạt được tại threshold đó: {closest_fpr*100:.2f}%")

# Dự đoán bằng nhãn lớp với ngưỡng tối ưu
y_pred_optimal = (probs >= target_threshold).astype(int)


### Confusion Matrix của mô hình tối ưu threshold

In [ ]:
# Tạo Confusion Matrix biểu diễn dự đoán tối ưu ngưỡng
cm = confusion_matrix(y_test, y_pred_optimal)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
disp.plot()
plt.title('Confusion Matrix - Threshold Optimized')
plt.show()


## So sánh LR From Scratch và LR `sklearn`

Ta so sánh hiệu năng của hai cách triển khai sử dụng các trọng số tối ưu từ Grid Search.

In [ ]:
best_params = lr_grid_search_f1.best_params_
print('Best params (F1-score):', best_params)

# ── From Scratch với best params ──────────────────────
scratch_best_pred = best_scratch_model.predict(X_test, threshold=0.5)
scratch_best_probs = best_scratch_model.predict_proba(X_test)

acc_scratch = accuracy_score(y_test, scratch_best_pred)
prec_scratch = precision_score(y_test, scratch_best_pred)
rec_scratch = recall_score(y_test, scratch_best_pred)
f1_scratch = f1_score(y_test, scratch_best_pred)

# ── sklearn với cùng best params ─────────────────────
# Để so sánh trực quan nhất, dùng mô hình sklearn cơ bản
sklearn_best = SklearnLR(max_iter=1000, random_state=42)
start = time.time()
sklearn_best.fit(X_train, y_train)
train_time_sklearn = time.time() - start

sklearn_best_pred = sklearn_best.predict(X_test)
sklearn_best_probs = sklearn_best.predict_proba(X_test)[:, 1]

acc_sklearn = accuracy_score(y_test, sklearn_best_pred)
prec_sklearn = precision_score(y_test, sklearn_best_pred)
rec_sklearn = recall_score(y_test, sklearn_best_pred)
f1_sklearn = f1_score(y_test, sklearn_best_pred)

# ── Bảng so sánh ─────────────────────────────────────
print('\n' + '='*60)
print(f"{'Metric':<15} {'From Scratch':>20} {'sklearn':>20}")
print('='*60)
print(f"{'Accuracy':<15} {acc_scratch:>20.4f} {acc_sklearn:>20.4f}")
print(f"{'Precision':<15} {prec_scratch:>20.4f} {prec_sklearn:>20.4f}")
print(f"{'Recall':<15} {rec_scratch:>20.4f} {rec_sklearn:>20.4f}")
print(f"{'F1 Score':<15} {f1_scratch:>20.4f} {f1_sklearn:>20.4f}")
print(f"{'Train Time (s)':<15} {train_time_scratch:>20.1f} {train_time_sklearn:>20.1f}")
print('='*60)

# So sánh sai lệch dự đoán xác suất
max_diff = np.max(np.abs(scratch_best_probs - sklearn_best_probs))
print(f'\nSai lệch dự đoán xác suất tối đa giữa 2 mô hình: {max_diff:.4f}')


## Feature Importance

Với Logistic Regression, độ quan trọng của các feature có thể được đánh giá dựa trên giá trị tuyệt đối của các hệ số (coefficients) hồi quy. Hệ số lớn biểu thị mức độ tác động mạnh mẽ của feature đó đến quyết định phân loại.

In [ ]:
coefs = sklearn_best.coef_[0]
n_features = X_train.shape[1]
feature_names = [f'feature_{i}' for i in range(n_features)]

# Trích xuất các index của hệ số lớn nhất theo giá trị tuyệt đối
sorted_idx = np.argsort(np.abs(coefs))[::-1]

print('Top 10 features quan trọng nhất (hệ số có trọng số lớn nhất):')
print(f"{'Rank':<6} {'Feature':<15} {'Weight (Coefficient)':>22}")
print('-' * 45)
for rank, idx in enumerate(sorted_idx[:10], 1):
    print(f"{rank:<6} {feature_names[idx]:<15} {coefs[idx]:>22.4f}")


## Kết luận

### Hướng triển khai Logistic Regression
- **From Scratch:** Tự xây dựng mô hình sử dụng hàm Sigmoid, hàm BCE loss và cập nhật Gradient Descent giúp hiểu sâu sắc cơ chế tối ưu hóa đằng sau mô hình hồi quy logistic tuyến tính.
- **Mô hình `sklearn.linear_model.LogisticRegression`** tối ưu hóa cao về mặt tính toán số học, có hỗ trợ các bộ giải cao cấp như `lbfgs` hoặc `liblinear`, mang lại tốc độ thực thi vượt trội trên lượng dữ liệu lớn.

### So sánh hiệu năng và sai lệch xác suất
Các chỉ số độ đo (Accuracy, Precision, Recall, F1) của phiên bản tự viết và thư viện vô cùng tiệm cận nhau. Sai lệch dự đoán xác suất tối đa cực kỳ nhỏ chứng minh tính đúng đắn trong các công thức toán học tự cài đặt.

### Vai trò của Threshold Tuning
Việc lựa chọn ngưỡng quyết định tối ưu đóng vai trò cực kỳ quan trọng trong thực tế. Bằng việc nâng ngưỡng xác suất lên cao (ngưỡng tối ưu khoảng 0.58), ta đã thành công giảm tỷ lệ cảnh báo nhầm (FPR) xuống mức cực thấp (dưới 1%), bảo vệ các thư email thông thường không bị đưa vào thư rác nhầm lẫn.

## Lưu model

In [ ]:
MODEL_DIR = './models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Lưu model from scratch
joblib.dump(best_scratch_model,
            os.path.join(MODEL_DIR, 'logistic_regression_scratch.pkl'))
print(f"Mô hình Logistic Regression Scratch đã được lưu tại "
      f"{os.path.join(MODEL_DIR, 'logistic_regression_scratch.pkl')}")

# Lưu model sklearn
joblib.dump(sklearn_best,
            os.path.join(MODEL_DIR, 'logistic_regression_sklearn.pkl'))
print(f"Mô hình Logistic Regression sklearn đã được lưu tại "
      f"{os.path.join(MODEL_DIR, 'logistic_regression_sklearn.pkl')}")
